In [3]:
import cv2
import math
import numpy as np
import easyocr
import imutils
from collections import defaultdict
from ultralytics import YOLO

In [ ]:
# Entrenar modelo de reconocimiento de matrículas
model = YOLO('yolo26x.pt')
results = model.train(data='uc3m-lp.yaml', epochs=70, imgsz=640)

In [ ]:
model = YOLO('runs/detect/50epochs/weights/best.pt')

cap = cv2.VideoCapture("video.MP4")

while cap.isOpened():
    success, frame = cap.read()

    if success:
        results = model.track(frame, verbose=False)
        frame = results[0].plot()
        cv2.imshow("YOLO", frame)
        
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
# Detección de personas, vehículos y matrículas SIN OCR

plates_model = YOLO('runs/detect/50epochs/weights/best.pt')
vehicle_model = YOLO('yolo26n.pt')

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck"]

filename = "video.MP4"
cap = cv2.VideoCapture(filename)
track_history = defaultdict(None)

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.MP4', fourcc, fps, (frame_width, frame_height))

with open("results.csv", "w") as file:
    file.write("fotograma;tipo_objeto;confianza;identificador_tracking;x1;y1;x2;y2;matrícula_en_su_caso;confianza;mx1;my1;mx2;my2\n")

frame_num = 0
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        frame_num += 1

        # Run YOLO inference on the frame
        results = vehicle_model.track(frame, persist=True, verbose=False, classes=[0,2,3,5,7], tracker="tracktrack.yaml")
        if results is not None:
            plates_res = plates_model(frame, verbose=False)
        else:
            plates_res = None

        #frame = plates_res[0].plot()

        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = list(map(int, box.xyxy[0]))

                confidence = math.ceil((box.conf[0]*100))/100
                cls = int(box.cls[0])

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                    track_history[track_id] = cls
                else:
                    track_id = ''

                # Colores distintos para personas y vehículos
                if cls == 0:
                    R = 20
                    G = 180
                    B = 20
                else:
                    R = 100
                    G = 0
                    B = 200

                csv_string = f"{frame_num};{classNames[cls]};{confidence};{track_id};{x1};{y1};{x2};{y2}"

                # Dibuja el contenedor y clase
                cv2.rectangle(frame, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(frame, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

                if cls != 0:
                    plate_found = False
                    for lpres in plates_res:
                        for plate in lpres.boxes:
                            lpx1, lpy1, lpx2, lpy2 = list(map(int, plate.xyxy[0]))
                            # Comprobar si la matrícula está dentro de la detección del vehículo
                            if lpx1 > x1 and lpx2 < x2 and lpy1 > y1 and lpy2 < y2:
                                plate_conf = math.ceil((plate.conf[0]*100))/100
                                cv2.rectangle(frame, (lpx1, lpy1), (lpx2, lpy2), (200, 50, 20), 3)
                                cv2.putText(frame, track_id + ' plate', [lpx1, lpy1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                                csv_string += f";true;{plate_conf};{lpx1};{lpy1};{lpx2};{lpy2}"
                                plate_found = True
                                break
                    if not plate_found:
                        csv_string += f";false;null;null;null;null;null"
                else:
                    csv_string += f";false;null;null;null;null;null"

                #print(csv_string)
                with open("results.csv", "a") as file:
                    file.write(csv_string + "\n")

        # Display the annotated frame
        out.write(frame)
        frame = cv2.resize(frame, (frame_width//2, frame_height//2), fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
        cv2.imshow("YOLO Inference", frame)
        #cv2.imshow("YOLO Inference", results[0].plot())

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Contar instancias de cada clase
with open("total.txt", "w") as file:
    for i, cls in enumerate(classNames):
        total = len(list(filter(lambda x: x[1] == i, track_history.items())))
        if total > 0:
            total_str = f"{cls}: {total}\n"
            print(total_str)
            file.write(total_str)

#results = model(filename, show=True)
cap.release()
out.release()
cv2.destroyAllWindows()

In [1]:
# Funcion de preprocesado de matrículas
def ocr_preprocess(img):
    img = cv2.resize(img, (1200, 400), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.blur(img, (3, 3))
    thresh = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 255, 1)
    
    # apply a distance transform which calculates the distance to the
    # closest zero pixel for each pixel in the input image
    dist = cv2.distanceTransform(thresh, cv2.DIST_L2, 5)
    
    # normalize the distance transform such that the distances lie in
    # the range [0, 1] and then convert the distance transform back to
    # an unsigned 8-bit integer in the range [0, 255]
    dist = cv2.normalize(dist, dist, 0, 1.0, cv2.NORM_MINMAX)
    dist = (dist * 255).astype("uint8")
    
    # threshold the distance transform using Otsu's method
    dist = cv2.threshold(dist, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    kernel = np.ones((5, 5), np.uint8)
    img = cv2.dilate(dist, kernel, iterations=1)
    img = cv2.erode(img, kernel, iterations=1)
    img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
    img = cv2.medianBlur(img, 3)

    # find contours in the opening image, then initialize the list of
    # contours which belong to actual characters that we will be OCR'ing
    cnts = cv2.findContours(img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    chars = []
    # loop over the contours
    for c in cnts:
    	# compute the bounding box of the contour
    	(x, y, w, h) = cv2.boundingRect(c)
    	# check if contour is at least 35px wide and 100px tall, and if
    	# so, consider the contour a digit
    	if w >= 35 and w <= 300 and h >= 100 and h <=300:
    		chars.append(c)

    # compute the convex hull of the characters
    if len(chars) == 0:
        return img
    chars = np.vstack([chars[i] for i in range(0, len(chars))])
    hull = cv2.convexHull(chars)
    # allocate memory for the convex hull mask, draw the convex hull on
    # the image, and then enlarge it via a dilation
    mask = np.zeros(img.shape[:2], dtype="uint8")
    cv2.drawContours(mask, [hull], -1, 255, -1)
    mask = cv2.dilate(mask, None, iterations=2)
    # take the bitwise of the opening image and the mask to reveal *just*
    # the characters in the image
    img = cv2.bitwise_and(img, img, mask=mask)
    
    return img

# Search for a valid plate in a string given by the OCR
import re
plate_re = re.compile(r'\d{4}[A-Z]{3}')
def search_plate(plate_str):
    res = plate_re.search(plate_str)
    if res is None:
        return ""
    else:
        return res.group()

In [4]:
# Detección de personas, vehículos y matrículas CON OCR (easyOCR)

plates_model = YOLO('runs/detect/100epochs/weights/best.pt')
vehicle_model = YOLO('yolo26n.pt')
ocr_reader = easyocr.Reader(['es', 'en'])
plate_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890"

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck"]

PLATE_FRAME_DEBUG = True

filename = "video.MP4"
cap = cv2.VideoCapture(filename)
track_history = defaultdict(None)

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.MP4', fourcc, fps, (frame_width, frame_height))

with open("results_ocr.csv", "w") as file:
    file.write("fotograma;tipo_objeto;confianza;identificador_tracking;x1;y1;x2;y2;matrícula_en_su_caso;confianza;mx1;my1;mx2;my2;text_matrícula\n")

frame_num = 0
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        frame_num += 1

        # Run YOLO inference on the frame
        results = vehicle_model.track(frame, persist=True, verbose=False, classes=[0,2,3,5,7], tracker="tracktrack.yaml")
        if results is not None:
            plates_res = plates_model(frame, verbose=False)
        else:
            plates_res = None

        #frame = plates_res[0].plot()

        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = list(map(int, box.xyxy[0]))

                confidence = math.ceil((box.conf[0]*100))/100
                cls = int(box.cls[0])

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                    track_history[track_id] = cls
                else:
                    track_id = ''

                # Colores distintos para personas y vehículos
                if cls == 0:
                    R = 20
                    G = 180
                    B = 20
                else:
                    R = 100
                    G = 0
                    B = 200

                csv_string = f"{frame_num};{classNames[cls]};{confidence};{track_id};{x1};{y1};{x2};{y2}"

                # Dibuja el contenedor y clase
                cv2.rectangle(frame, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(frame, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

                if cls != 0:
                    plate_found = False
                    for lpres in plates_res:
                        for plate in lpres.boxes:
                            lpx1, lpy1, lpx2, lpy2 = list(map(int, plate.xyxy[0]))
                            # Comprobar si la matrícula está dentro de la detección del vehículo
                            if lpx1 > x1 and lpx2 < x2 and lpy1 > y1 and lpy2 < y2:
                                plate_conf = math.ceil((plate.conf[0]*100))/100
                                plate_frame = frame[lpy1:lpy2, lpx1:lpx2].copy()
                                cv2.rectangle(frame, (lpx1, lpy1), (lpx2, lpy2), (200, 50, 20), 3)
                                cv2.putText(frame, track_id + ' plate', [lpx1, lpy1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                                csv_string += f";true;{plate_conf};{lpx1};{lpy1};{lpx2};{lpy2}"
                                # Preprocesado de la matrícula para el OCR
                                plate_frame = ocr_preprocess(plate_frame)
                                if PLATE_FRAME_DEBUG:
                                    cv2.imshow("Id: " + str(track_id), plate_frame)
                                #ocr_res = ocr_reader.recognize(plate_frame)
                                ocr_res= ocr_reader.readtext(plate_frame, allowlist=plate_chars)
                                ocr_text = "".join(list(map(lambda x: x[1], ocr_res)))
                                if len(ocr_res) > 0:
                                    ocr_text = search_plate(ocr_text)
                                    if ocr_text != "":
                                        csv_string += f";{ocr_text}"
                                    else:
                                        csv_string += f";null"
                                plate_found = True
                                break
                    if not plate_found:
                        csv_string += f";false;null;null;null;null;null;null"
                else:
                    csv_string += f";false;null;null;null;null;null;null"

                #print(csv_string)
                with open("results_ocr.csv", "a") as file:
                    file.write(csv_string + "\n")

        # Display the annotated frame
        out.write(frame)
        frame = cv2.resize(frame, (frame_width//2, frame_height//2), fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)
        cv2.imshow("YOLO Inference", frame)
        #cv2.imshow("YOLO Inference", results[0].plot())

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Contar instancias de cada clase
with open("total.txt", "w") as file:
    for i, cls in enumerate(classNames):
        total = len(list(filter(lambda x: x[1] == i, track_history.items())))
        if total > 0:
            total_str = f"{cls}: {total}\n"
            print(total_str)
            file.write(total_str)

#results = model(filename, show=True)
cap.release()
out.release()
cv2.destroyAllWindows()


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/Gerad/.conda/envs/VC_P4/lib/python3.9/site-packages/cv2/qt/plugins"


person: 13

car: 34

motorbike: 1

truck: 1



In [9]:
cv2.destroyAllWindows()

In [1]:
# Preparación del modelo VLM
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
Device = "cuda"  # or "cpu"

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct")
#flash_attention_2
model = AutoModelForImageTextToText.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct",
                                                dtype=torch.bfloat16,
                                                _attn_implementation="eager" if Device == "cuda" else "eager").to(Device)

from PIL import Image
from transformers.image_utils import load_image
from matplotlib import pyplot as plt

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Concatenate and return every character"}
        ]
    },
]

vlm_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
def test_vlm(img):
    inputs = processor(text=vlm_prompt, images=[img], return_tensors="pt")
    inputs = inputs.to(Device)

    generated_ids = model.generate(**inputs, max_new_tokens=10)
    generated_texts = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return generated_texts[0].split(" ")[-1].strip()

# Preparación de easyOCR
import easyocr
import cv2
import imutils
import numpy as np

def ocr_preprocess(img):
    img = cv2.resize(img, (1200, 400), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.blur(img, (3, 3))
    thresh = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 255, 1)
    
    # apply a distance transform which calculates the distance to the
    # closest zero pixel for each pixel in the input image
    dist = cv2.distanceTransform(thresh, cv2.DIST_L2, 5)
    
    # normalize the distance transform such that the distances lie in
    # the range [0, 1] and then convert the distance transform back to
    # an unsigned 8-bit integer in the range [0, 255]
    dist = cv2.normalize(dist, dist, 0, 1.0, cv2.NORM_MINMAX)
    dist = (dist * 255).astype("uint8")
    
    # threshold the distance transform using Otsu's method
    dist = cv2.threshold(dist, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    
    kernel = np.ones((5, 5), np.uint8)
    img = cv2.dilate(dist, kernel, iterations=1)
    img = cv2.erode(img, kernel, iterations=1)
    img = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    #img = cv2.medianBlur(img, 3)

    # find contours in the opening image, then initialize the list of
    # contours which belong to actual characters that we will be OCR'ing
    cnts = cv2.findContours(img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    chars = []
    # loop over the contours
    for c in cnts:
    	# compute the bounding box of the contour
    	(x, y, w, h) = cv2.boundingRect(c)
    	# check if contour is at least 35px wide and 100px tall, and if
    	# so, consider the contour a digit
    	if w >= 35 and w <= 300 and h >= 100 and h <=300:
    		chars.append(c)

    # compute the convex hull of the characters
    if len(chars) == 0:
        return img
    chars = np.vstack([chars[i] for i in range(0, len(chars))])
    hull = cv2.convexHull(chars)
    # allocate memory for the convex hull mask, draw the convex hull on
    # the image, and then enlarge it via a dilation
    mask = np.zeros(img.shape[:2], dtype="uint8")
    cv2.drawContours(mask, [hull], -1, 255, -1)
    mask = cv2.dilate(mask, None, iterations=2)
    # take the bitwise of the opening image and the mask to reveal *just*
    # the characters in the image
    img = cv2.bitwise_and(img, img, mask=mask)
    
    return img

ocr_reader = easyocr.Reader(['es', 'en'])
plate_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890"

def test_ocr(img):
    return "".join(ocr_reader.readtext(img, allowlist=plate_chars, detail=0)).strip()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
# Comparación de OCR vs VLM

from ultralytics import YOLO
plates_model = YOLO('runs/detect/100epochs/weights/best.pt')

import time

# En este fichero se encuentra la asociación entre las imágenes y el texto de la matrícula
plates_file = "plates.txt"
images_dir = "UC3M-LP-yolo/LP/images/train/"

images_processed = 0
vlm_hits = 0
ocr_hits = 0

SHOW_IMAGES=False

with open("OCRvcVLM.csv", "w") as file:
    file.write("ID imagen;Matrícula;Matrícula VLM;Tiempo VLM;Matrícula OCR;Tiempo OCR\n")

with open(plates_file, "r") as pfile:
    for line in pfile:
        image_id, plate_str = line.strip().split("-")
        image_name = f"{images_dir}{image_id}.jpg"
        print(f"------------------------------------")
        print(f"Fichero: {image_name} {plate_str}")
        #image = Image.open(image_name)
        image = cv2.imread(image_name)
        
        # Preprocesado de la imagen
        plate_res = plates_model(image, verbose=False)[0].boxes
        plate_res = list(filter(lambda x: float(x.conf[0]) > 0.7, plate_res))
        if len(plate_res) == 0:
            continue
        plate_res = plate_res[0]
        lpx1, lpy1, lpx2, lpy2 = list(map(int, plate_res.xyxy[0]))
        image = image[lpy1:lpy2, lpx1:lpx2]

        vlm_time = time.time()
        vlm_text = test_vlm(image)
        vlm_time = time.time() - vlm_time
        if vlm_text == plate_str:
            vlm_hits += 1

        if SHOW_IMAGES:
            plt.subplot(211)
            plt.imshow(image)
            plt.axis("off")
            image = ocr_preprocess(image)
            plt.subplot(212)
            plt.imshow(image, cmap='Greys')
            plt.axis("off")
            plt.show()

        ocr_time = time.time()
        ocr_text = test_ocr(image)
        ocr_time = time.time() - ocr_time
        if ocr_text == plate_str:
            ocr_hits += 1

        print(f"Matrícula real: {plate_str}")
        print(f"")
        print(f"Matrícula según smolVLM: {vlm_text}")
        print(f"Tiempo de inferencia: {vlm_time:.5f}")
        print(f"")
        print(f"Matrícula según easyOCR: {ocr_text}")
        print(f"Tiempo de inferencia: {ocr_time:.5f}")
        print(f"")
        with open("OCRvcVLM.csv", "a") as file:
            file.write(f"{image_id};{plate_str};{vlm_text};{vlm_time};{ocr_text};{ocr_time}\n")
        
        # process image
        images_processed += 1

print(f"Aciertos VLM: {vlm_hits}")
print(f"Aciertos OCR: {ocr_hits}")

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00023.jpg 054PX
Matrícula real: 054PX

Matrícula según smolVLM: 054PX
Tiempo de inferencia: 1.63476

Matrícula según easyOCR: PX103L
Tiempo de inferencia: 0.04067

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00025.jpg 496TM
Matrícula real: 496TM

Matrícula según smolVLM: ming
Tiempo de inferencia: 0.81930

Matrícula según easyOCR: 
Tiempo de inferencia: 0.01363

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00027.jpg 867KT
Matrícula real: 867KT

Matrícula según smolVLM: 867KTI
Tiempo de inferencia: 0.89271

Matrícula según easyOCR: 86T7KT
Tiempo de inferencia: 0.02145

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00028.jpg 115GX
Matrícula real: 115GX

Matrícula según smolVLM: 11511GX
Tiempo de inferencia: 0.93959

Matrícula según easyOCR: 10IE5GX
Tiempo de inferencia: 0.00886

------------------------------------
Fichero: U

The channel dimension is ambiguous. Got image shape (4, 14, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Matrícula real: 193RZ

Matrícula según smolVLM: 193RZ
Tiempo de inferencia: 0.89912

Matrícula según easyOCR: 19B3RZ2U
Tiempo de inferencia: 0.01463

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00054.jpg 641CR
Matrícula real: 641CR

Matrícula según smolVLM: e
Tiempo de inferencia: 0.47025

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00220

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00056.jpg 961MV
Matrícula real: 961MV

Matrícula según smolVLM: 101010101
Tiempo de inferencia: 0.64507

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00223

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00057.jpg 441XZ
Matrícula real: 441XZ

Matrícula según smolVLM: 4612
Tiempo de inferencia: 0.92291

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00229

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00058.jpg 922ST
Matrícula real: 922ST

Matrícula según smolVLM: 92251
Tiempo 

The channel dimension is ambiguous. Got image shape (4, 17, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Matrícula real: 513BR

Matrícula según smolVLM: 511322
Tiempo de inferencia: 0.92747

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00234

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00062.jpg 750CR
Matrícula real: 750CR

Matrícula según smolVLM: e
Tiempo de inferencia: 0.47247

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00231

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00063.jpg 252GV
Matrícula real: 252GV

Matrícula según smolVLM: 2.52
Tiempo de inferencia: 0.57052

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00227

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00064.jpg 151ZY
Matrícula real: 151ZY

Matrícula según smolVLM: 151010101
Tiempo de inferencia: 1.00805

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00235

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00065.jpg 555XV
Matrícula real: 555XV

Matrícula según smolVLM: yy
Tiempo de inferen

The channel dimension is ambiguous. Got image shape (4, 17, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Matrícula real: 056BV

Matrícula según smolVLM: a
Tiempo de inferencia: 0.49043

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00219

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00071.jpg 599FW
Matrícula real: 599FW

Matrícula según smolVLM: e
Tiempo de inferencia: 0.62550

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00220

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00072.jpg 907CV
Matrícula real: 907CV

Matrícula según smolVLM: 507
Tiempo de inferencia: 0.86806

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00234

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00073.jpg 723PV
Matrícula real: 723PV

Matrícula según smolVLM: 7236
Tiempo de inferencia: 0.54846

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00218

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00080.jpg 303NT
Matrícula real: 303NT

Matrícula según smolVLM: 303
Tiempo de inferencia: 0.521

The channel dimension is ambiguous. Got image shape (4, 18, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Matrícula real: 676LM

Matrícula según smolVLM: 676_L_W
Tiempo de inferencia: 0.60784

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00229

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00134.jpg 548GC
Matrícula real: 548GC

Matrícula según smolVLM: e
Tiempo de inferencia: 0.48467

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00223

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00135.jpg 441TB
Matrícula real: 441TB

Matrícula según smolVLM: a
Tiempo de inferencia: 0.48744

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00228

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00136.jpg 665DV
Matrícula real: 665DV

Matrícula según smolVLM: 665500000
Tiempo de inferencia: 0.61607

Matrícula según easyOCR: 
Tiempo de inferencia: 0.00225

------------------------------------
Fichero: UC3M-LP-yolo/LP/images/train/00140.jpg 473FX
Matrícula real: 473FX

Matrícula según smolVLM: 473F1
Tiempo de infere